In [11]:
%pip install -q pandas pyarrow

import csv
from pathlib import Path

import pandas as pd

DATA = Path("data")  # data/ folder next to the notebook
DATA.mkdir(exist_ok=True)

In [12]:
# ORCAS click logs (~330 MB) and MS MARCO documents (~8.4 GB) go to data/; -C - resumes a partial download.
!curl -fL -C - -o "{DATA}/orcas.tsv.gz" https://msmarco.z22.web.core.windows.net/msmarcoranking/orcas.tsv.gz
!curl -fL -C - -o "{DATA}/msmarco-docs.tsv.gz" https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-docs.tsv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  314M  100  314M    0     0  31.0M      0  0:00:10  0:00:10 --:--:-- 29.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8054M  100 8054M    0     0  30.7M      0  0:04:22  0:04:22 --:--:-- 29.5M


In [13]:
# pandas reads .gz directly. QUOTE_NONE and keep_default_na stop stray quotes and queries like "null" from being misread.
orcas = pd.read_csv(
    DATA / "orcas.tsv.gz",
    sep="\t",
    names=["qid", "query", "did", "url"],
    dtype=str,
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)
print(orcas.shape)
orcas.head()

(18823602, 4)


,qid,query,did,url
0,9265503,github,D1265400,https://desktop.github.com/
1,9265503,github,D3438005,https://github.com/
2,6832981,youtube,D2923232,https://au.youtube.com/
3,9571352,!,D1238374,https://www.englishclub.com/writing/punctuatio...
4,9571352,!,D889000,http://nationalpunctuationday.com/exclamationp...


In [14]:
# usecols skips the page body, which is most of the file and never used.
docs = pd.read_csv(
    DATA / "msmarco-docs.tsv.gz",
    sep="\t",
    names=["did", "url", "title", "body"],
    usecols=["did", "url", "title"],
    dtype=str,
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)
print(docs.shape)
docs.head()

(3213835, 3)


,did,url,title
0,D1555982,https://answers.yahoo.com/question/index?qid=2...,The hot glowing surfaces of stars emit energy ...
1,D301595,http://childparenting.about.com/od/physicalemo...,Developmental Milestones and Your 8-Year-Old C...
2,D1359209,http://visihow.com/Check_for_Lice_Nits,Check for Lice Nits
3,D2147834,http://www.nytimes.com/2010/01/05/business/glo...,Dubai Opens a Tower to Beat All
4,D1568809,http://www.realtor.com/realestateandhomes-sear...,"Coulterville, CA Real Estate & Homes for Sale"


In [15]:
# Each click maps to one document, so validate many-to-one.
df = orcas.merge(docs[["did", "title"]], on="did", validate="many_to_one")

18823602 -> 18823602


In [1]:
# Look at the word-count distribution before picking a cap.
df["query"] = df["query"].str.strip()
df["title"] = df["title"].str.strip()

query_words = df["query"].str.split().str.len()
title_words = df["title"].str.split().str.len()

print(query_words.describe())
print(title_words.describe())

NameError: name 'df' is not defined

In [ ]:
# Set this from the describe() output above: just past where the long tail starts.
MAX_WORDS = 30

has_text = df["query"].str.contains(r"[^\W_]") & df["title"].str.contains(r"[^\W_]")
short_enough = (query_words <= MAX_WORDS) & (title_words <= MAX_WORDS)

df = df[has_text & short_enough].reset_index(drop=True)
print(len(df))
df.head()